# 07 RAG Retrieval Engine

This notebook implements the Retrieval-Augmented Generation (RAG) data pipeline. 
Since the test set IDs were dropped during model training, we recreate the exact `train_test_split` (using the same random seed) to preserve `hadm_id`. This allows us to map patient-specific SHAP values to their clinical text for LLM generation.

**Steps:**
1. Recreate `X_test` with `hadm_id`.
2. Load the trained model and SHAP explainer from `model_artifacts.pkl`.
3. Load the clinical notes (`HPI.json` and `RAG_data_summary.json`).
4. Build a retrieval function to construct a prompt combining clinical context with AI risk drivers.


In [15]:
import duckdb
import pandas as pd
import numpy as np
import pickle
import json
import shap
import warnings
warnings.filterwarnings('ignore')

from utils import get_db_connection


In [16]:
# Recreate X_test while preserving hadm_id
print("Connecting to DB to load model_features...")
con = get_db_connection()
df = con.execute("SELECT * FROM model_features").fetchdf()

# Add lab CHANGE features (last - first)
LABS = [
    'creatinine', 'urea_nitrogen', 'sodium', 'potassium', 'glucose',
    'hemoglobin', 'white_blood_cells', 'platelet_count', 'bicarbonate',
    'calcium_total', 'inrpt', 'ptt', 'troponin_t',
    'creatine_kinase_mb_isoenzyme'
]
for lab in LABS:
    df[f'{lab}_change'] = df[f'{lab}_last'] - df[f'{lab}_first']

# Fill missing values
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('readmitted_30d')
num_cols.remove('hadm_id')
num_cols.remove('subject_id')

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

cat_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

# Encode categorical columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Train/Test Split (keeping hadm_id)
from sklearn.model_selection import train_test_split

# Drop subject_id but KEEP hadm_id this time
drop_cols = ['subject_id', 'readmitted_30d']
X = df_encoded.drop(columns=drop_cols)
y = df_encoded['readmitted_30d']

X_train_id, X_test_id, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Extract hadm_ids and drop them from the feature set
test_hadm_ids = X_test_id['hadm_id'].values
X_test = X_test_id.drop(columns=['hadm_id'])

print(f"Recreated X_test with shape: {X_test.shape}")
print(f"Number of test hadm_ids: {len(test_hadm_ids)}")
con.close()


Connecting to DB to load model_features...
Connected to DuckDB at: /Users/sameer/Documents/DataScience_Capstone_Project/Capstone_Healthcare_Decision_Intelligence_Agent/dataset/hf_project.duckdb
Recreated X_test with shape: (902, 137)
Number of test hadm_ids: 902


In [17]:
# Load Model Artifacts
artifacts_path = '/Users/sameer/Desktop/AIheathAgent/Capstone_Healthcare_Decision_Intelligence_Agent/model_artifacts.pkl'
with open(artifacts_path, 'rb') as f:
    artifacts = pickle.load(f)

scaler = artifacts['scaler']
selector = artifacts['selector']
model = artifacts['model']
selected_features = artifacts['selected_features']

# Preprocess X_test
X_test_selected = selector.transform(X_test)
X_test_scaled = scaler.transform(X_test_selected)
X_test_df = pd.DataFrame(X_test_scaled, columns=selected_features)

# Generate SHAP values
print("Calculating SHAP values for the test set...")
explainer = shap.LinearExplainer(model, X_test_df)
shap_values = explainer(X_test_df)
print("SHAP values calculated successfully.")

# Get probabilities
y_pred_proba = model.predict_proba(X_test_df)[:, 1]


Calculating SHAP values for the test set...
SHAP values calculated successfully.


In [18]:
# Load RAG Data
with open('/Users/sameer/Desktop/AIheathAgent/Capstone_Healthcare_Decision_Intelligence_Agent/dataset/HPI.json', 'r') as f:
    hpi_list = json.load(f)
with open('/Users/sameer/Desktop/AIheathAgent/Capstone_Healthcare_Decision_Intelligence_Agent/dataset/RAG_data_summary.json', 'r') as f:
    rag_list = json.load(f)

# Convert to dictionaries for O(1) lookup
hpi_data = {list(item.keys())[0]: list(item.values())[0] for item in hpi_list}
rag_data = {list(item.keys())[0]: list(item.values())[0] for item in rag_list}

print(f"Loaded {len(hpi_data)} HPI records and {len(rag_data)} RAG summaries.")


Loaded 1261 HPI records and 1261 RAG summaries.


In [19]:
# RAG Retrieval Function
def get_patient_context(hadm_id, num_features=3):
    hadm_id_str = str(hadm_id)
    
    # 1. Find patient in test set
    if hadm_id not in test_hadm_ids:
        return f"Error: Patient {hadm_id} not found in the test set."
    
    idx = np.where(test_hadm_ids == hadm_id)[0][0]
    
    # 2. Get Model Prediction
    risk_prob = y_pred_proba[idx]
    
    # 3. Get Top SHAP Drivers
    patient_shap = shap_values.values[idx]
    feature_names = shap_values.feature_names
    
    # Sort features by SHAP value (descending = most increasing risk)
    sorted_idx = np.argsort(patient_shap)[::-1]
    top_drivers = [(feature_names[i], patient_shap[i]) for i in sorted_idx[:num_features] if patient_shap[i] > 0]
    
    drivers_str = "\n".join([f"- {feat} (Impact score: {val:.4f})" for feat, val in top_drivers])
    if not drivers_str:
        drivers_str = "No significant risk-increasing factors found."
        
    # 4. Get Clinical Summary
    clinical_summary = rag_data.get(hadm_id_str, "No clinical summary available.")
    
    # 5. Format Prompt Context
    prompt_context = f"""--- PATIENT CLINICAL HISTORY ---
{clinical_summary}

--- AI MODEL PREDICTION ---
Predicted Readmission Risk: {risk_prob:.1%}

--- KEY RISK DRIVERS (SHAP) ---
The model identified the following key factors driving this risk:
{drivers_str}
"""
    return prompt_context

def generate_explanation_placeholder(prompt_context):
    """
    Placeholder for the LLM API call. 
    You will plug in OpenAI or Gemini here later.
    """
    system_prompt = """You are an expert clinical assistant. Review the provided patient summary and the predictive model's risk drivers. 
1. Provide a concise, clinician-friendly explanation of why the model predicts this risk.
2. Suggest 3 actionable clinical precautions or next steps based on the drivers."""
    
    full_prompt = f"{system_prompt}\n\n{prompt_context}"
    
    # Example of where the LLM call goes:
    # response = openai.ChatCompletion.create(...)
    # return response.choices[0].message.content
    
    return "LLM Generation Pending... (Plug in your API here)"

# Test the function on the first available patient
test_patient = test_hadm_ids[0]
context = get_patient_context(test_patient)
print("=== RAG PROMPT CONTEXT REVEAL ===\n")
print(context)


=== RAG PROMPT CONTEXT REVEAL ===

--- PATIENT CLINICAL HISTORY ---
No clinical summary available.

--- AI MODEL PREDICTION ---
Predicted Readmission Risk: 32.1%

--- KEY RISK DRIVERS (SHAP) ---
The model identified the following key factors driving this risk:
- bicarbonate_min (Impact score: 0.1681)
- hemoglobin_min (Impact score: 0.0951)
- potassium_last (Impact score: 0.0703)



In [20]:
# Find patients in test set WITH clinical summaries and run the full RAG pipeline

# Cross-reference test_hadm_ids with RAG data
rag_ids = set(rag_data.keys())
matched_ids = [hid for hid in test_hadm_ids if str(hid) in rag_ids]
print(f"Test patients with RAG summaries available: {len(matched_ids)} out of {len(test_hadm_ids)}")

# ------- Demo: show full prompt context for 3 patients --------
for hadm_id in matched_ids[:3]:
    print("\n" + "="*70)
    print(f"PATIENT hadm_id: {hadm_id}")
    print("="*70)
    context = get_patient_context(hadm_id)
    print(context)
    explanation = generate_explanation_placeholder(context)
    print("--- [PLACEHOLDER LLM OUTPUT] ---")
    print(explanation)

# -------- Save prompt context to JSON for LLM step --------
import os
output_path = os.path.join('/Users/sameer/Desktop/AIheathAgent/Capstone_Healthcare_Decision_Intelligence_Agent', 'rag_prompts.json')
rag_output = []
for hadm_id in matched_ids:
    ctx = get_patient_context(hadm_id)
    rag_output.append({'hadm_id': int(hadm_id), 'prompt_context': ctx})

with open(output_path, 'w') as f:
    json.dump(rag_output, f, indent=2)
print(f"\nSaved {len(rag_output)} patient prompt contexts to rag_prompts.json")


Test patients with RAG summaries available: 241 out of 902

PATIENT hadm_id: 22903651
--- PATIENT CLINICAL HISTORY ---
assistant

The patient is a female with a history of percutaneous coronary intervention (PCI) with a bare metal stent (BMS) in the left circumflex artery (LCx) and left anterior descending artery (LAD) due to coronary artery disease. She has been experiencing episodes of resting chest pain, which resolved with medication, and her electrocardiogram (ECG) showed T-wave inversion in septal leads. The patient's troponin levels were mildly elevated, and she was started on antiplatelet and anticoagulant medications. However, she reports a history of exertional angina, with her pain-free walking distance decreasing over the past few months. The patient's physical examination revealed a regular rhythm, but her blood pressure is slightly elevated, and her left arm has reduced pulses in the dorsal pedal. The laboratory test results indicate an elevated fasting blood glucose leve